In [2]:
# ============================================================
# SETUP
# ============================================================
!pip install -q optuna xgboost lightgbm

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
import optuna
import json
import pickle
import os
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

BASE = '/content/drive/MyDrive/siap'

freq_train = pd.read_csv(f'{BASE}/data/freq_train_v2.csv')
freq_test  = pd.read_csv(f'{BASE}/data/freq_test_v2.csv')
sev_full   = pd.read_csv(f'{BASE}/data/sev_full_v2.csv')

with open(f'{BASE}/data/feature_cols_v2.json') as f:
    X_cols = json.load(f)

print(f"Train freq : {freq_train.shape} | Test freq : {freq_test.shape}")
print(f"Sev full   : {sev_full.shape}")
print(f"GPU dispo  : {xgb.build_info().get('USE_CUDA', 'unknown')}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 11.8 MB/s eta 0:00:00
Mounted at /content/drive
Train freq : (531528, 46) | Test freq : (132882, 46)
Sev full   : (26340, 46)
GPU dispo  : True


In [3]:
# ============================================================
# FONCTION COMMUNE — DÉVIANCE POISSON
# ============================================================
def poisson_deviance(y_true, y_pred):
    eps = 1e-9
    y_pred = np.clip(y_pred, eps, None)
    terme = np.where(y_true == 0, 0, y_true * np.log(y_true / y_pred))
    return 2 * np.mean(terme - (y_true - y_pred))

# Préparation des données fréquence
X_train = freq_train[X_cols].astype(float)
y_train = freq_train['ClaimNb'].values
exposure_train = freq_train['Exposure'].values

X_test = freq_test[X_cols].astype(float)
y_test = freq_test['ClaimNb'].values
exposure_test = freq_test['Exposure'].values

# Split interne train/validation pour le tuning (80/20 du train)
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val, exp_tr, exp_val = train_test_split(
    X_train, y_train, exposure_train,
    test_size=0.2, random_state=42,
    stratify=(y_train > 0).astype(int)
)

In [4]:
# ============================================================
# BLOC 1 — XGBOOST POISSON (FRÉQUENCE) AVEC OFFSET + OPTUNA
# ============================================================
# Méthode de l'offset : XGBoost en objectif 'count:poisson'
# prédit exp(raw_score). On ajoute log(Exposure) au raw_score
# via base_margin → mu_predit = Exposure * exp(X @ f(X))
# C'est l'équivalent exact de l'offset du GLM.

dtrain_xgb = xgb.DMatrix(X_tr, label=y_tr, base_margin=np.log(exp_tr))
dval_xgb   = xgb.DMatrix(X_val, label=y_val, base_margin=np.log(exp_val))
dtrain_full_xgb = xgb.DMatrix(X_train, label=y_train, base_margin=np.log(exposure_train))
dtest_xgb  = xgb.DMatrix(X_test, label=y_test, base_margin=np.log(exposure_test))

def objective_xgb_freq(trial):
    params = {
        'objective': 'count:poisson',
        'tree_method': 'hist',
        'device': 'cuda',
        'eval_metric': 'poisson-nloglik',
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 50),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
    }

    model = xgb.train(
        params, dtrain_xgb,
        num_boost_round=500,
        evals=[(dval_xgb, 'val')],
        early_stopping_rounds=30,
        verbose_eval=False
    )

    pred_val = model.predict(dval_xgb)
    return poisson_deviance(y_val, pred_val)

study_xgb_freq = optuna.create_study(direction='minimize')
study_xgb_freq.optimize(objective_xgb_freq, n_trials=25, show_progress_bar=True)

print(f"Meilleure déviance (val) : {study_xgb_freq.best_value:.4f}")
print(f"Meilleurs params : {study_xgb_freq.best_params}")

  0%|          | 0/25 [00:00<?, ?it/s]

Meilleure déviance (val) : 0.3014
Meilleurs params : {'max_depth': 7, 'learning_rate': 0.07346284099029854, 'subsample': 0.872778117615111, 'colsample_bytree': 0.6988905794621478, 'min_child_weight': 6, 'reg_alpha': 0.0010103395092997858, 'reg_lambda': 0.03349188415078882}


In [5]:
# ============================================================
# BLOC 2 — RÉENTRAÎNEMENT FINAL XGBOOST FRÉQUENCE (TOUT LE TRAIN)
# ============================================================
best_params_xgb_freq = study_xgb_freq.best_params.copy()
best_params_xgb_freq.update({
    'objective': 'count:poisson',
    'tree_method': 'hist',
    'device': 'cuda',
    'eval_metric': 'poisson-nloglik',
})

# On garde un petit hold-out pour l'early stopping final
xgb_freq_final = xgb.train(
    best_params_xgb_freq, dtrain_xgb,
    num_boost_round=1000,
    evals=[(dval_xgb, 'val')],
    early_stopping_rounds=30,
    verbose_eval=False
)

print(f"Nombre d'arbres retenus : {xgb_freq_final.best_iteration}")

# Prédictions test
pred_xgb_freq_test = xgb_freq_final.predict(dtest_xgb)

dev_xgb_freq = poisson_deviance(y_test, pred_xgb_freq_test)
freq_obs  = y_test.sum() / exposure_test.sum()
freq_pred_xgb = pred_xgb_freq_test.sum() / exposure_test.sum()

print(f"Déviance Poisson XGBoost (test) : {dev_xgb_freq:.4f}")
print(f"Fréquence observée (test)        : {freq_obs:.4f}")
print(f"Fréquence prédite XGBoost (test) : {freq_pred_xgb:.4f}")

Nombre d'arbres retenus : 414
Déviance Poisson XGBoost (test) : 0.3044
Fréquence observée (test)        : 0.0998
Fréquence prédite XGBoost (test) : 0.0992


In [6]:
# ============================================================
# BLOC 3 — LIGHTGBM POISSON (FRÉQUENCE) AVEC OFFSET + OPTUNA
# ============================================================
# LightGBM : même logique d'offset via init_score = log(Exposure)
# Attention : par défaut le pip lightgbm n'est PAS compilé GPU,
# on reste donc en CPU pour LightGBM (rapide quand même avec
# histogram-based learning), GPU réservé à XGBoost.

dtrain_lgb = lgb.Dataset(X_tr, label=y_tr, init_score=np.log(exp_tr))
dval_lgb   = lgb.Dataset(X_val, label=y_val, init_score=np.log(exp_val), reference=dtrain_lgb)

def objective_lgb_freq(trial):
    params = {
        'objective': 'poisson',
        'metric': 'poisson',
        'verbosity': -1,
        'feature_pre_filter': False,
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'num_leaves': trial.suggest_int('num_leaves', 15, 255),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': 1,
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-3, 10, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-3, 10, log=True),
    }

    model = lgb.train(
        params, dtrain_lgb,
        num_boost_round=500,
        valid_sets=[dval_lgb],
        callbacks=[lgb.early_stopping(30, verbose=False)]
    )

    # predict() retourne exp(raw_score) ; il faut rajouter l'offset
    # manuellement car init_score n'est pas réappliqué automatiquement
    raw_val = model.predict(X_val, raw_score=True)
    pred_val = np.exp(raw_val + np.log(exp_val))
    return poisson_deviance(y_val, pred_val)

study_lgb_freq = optuna.create_study(direction='minimize')
study_lgb_freq.optimize(objective_lgb_freq, n_trials=25, show_progress_bar=True)

print(f"Meilleure déviance (val) : {study_lgb_freq.best_value:.4f}")
print(f"Meilleurs params : {study_lgb_freq.best_params}")

  0%|          | 0/25 [00:00<?, ?it/s]

Meilleure déviance (val) : 0.3015
Meilleurs params : {'max_depth': 8, 'num_leaves': 172, 'learning_rate': 0.042737808434171694, 'feature_fraction': 0.7052211941486577, 'bagging_fraction': 0.6929435042325808, 'min_child_samples': 87, 'lambda_l1': 0.005465418836115895, 'lambda_l2': 0.0012154784758213058}


In [7]:
# ============================================================
# BLOC 4 — RÉENTRAÎNEMENT FINAL LIGHTGBM FRÉQUENCE
# ============================================================
dtrain_full_lgb = lgb.Dataset(X_train, label=y_train, init_score=np.log(exposure_train))

best_params_lgb_freq = study_lgb_freq.best_params.copy()
best_params_lgb_freq.update({'objective': 'poisson', 'metric': 'poisson', 'verbosity': -1})

lgb_freq_final = lgb.train(
    best_params_lgb_freq, dtrain_lgb,
    num_boost_round=1000,
    valid_sets=[dval_lgb],
    callbacks=[lgb.early_stopping(30, verbose=False)]
)

raw_test = lgb_freq_final.predict(X_test, raw_score=True)
pred_lgb_freq_test = np.exp(raw_test + np.log(exposure_test))

dev_lgb_freq = poisson_deviance(y_test, pred_lgb_freq_test)
freq_pred_lgb = pred_lgb_freq_test.sum() / exposure_test.sum()

print(f"Déviance Poisson LightGBM (test) : {dev_lgb_freq:.4f}")
print(f"Fréquence prédite LightGBM (test) : {freq_pred_lgb:.4f}")

Déviance Poisson LightGBM (test) : 0.3042
Fréquence prédite LightGBM (test) : 0.0994


In [8]:
# ============================================================
# BLOC 5 — COMPARAISON FRÉQUENCE : GLM vs XGBoost vs LightGBM
# ============================================================
comparaison_freq = pd.DataFrame({
    'Modèle': ['GLM Poisson', 'XGBoost', 'LightGBM'],
    'Déviance test': [0.3199, dev_xgb_freq, dev_lgb_freq],
    'Fréquence prédite': [0.0995, freq_pred_xgb, freq_pred_lgb],
})
comparaison_freq['Fréquence observée'] = freq_obs
print(comparaison_freq)
# Une déviance plus basse = meilleur modèle
# Mais attention : si la fréquence prédite dérive trop de l'observée,
# le modèle n'est plus "balancé" → problème pour la tarification

        Modèle  Déviance test  Fréquence prédite  Fréquence observée
0  GLM Poisson       0.319900           0.099500            0.099819
1      XGBoost       0.304371           0.099186            0.099819
2     LightGBM       0.304235           0.099357            0.099819


In [9]:
# ============================================================
# PRÉPARATION DONNÉES SÉVÉRITÉ
# ============================================================
X_sev = sev_full[X_cols].astype(float)
y_sev = sev_full['ClaimAmount'].values

X_sev_tr, X_sev_val, y_sev_tr, y_sev_val = train_test_split(
    X_sev, y_sev, test_size=0.2, random_state=42
)
X_sev_train_full, X_sev_test, y_sev_train_full, y_sev_test = train_test_split(
    X_sev, y_sev, test_size=0.2, random_state=42
)
# (même split que le GLM pour comparabilité — random_state=42, test_size=0.2
#  appliqué directement sur sev_full comme au script 3)

In [10]:
# ============================================================
# BLOC 6 — XGBOOST GAMMA (SÉVÉRITÉ) AVEC OPTUNA
# ============================================================
dtrain_sev = xgb.DMatrix(X_sev_tr, label=y_sev_tr)
dval_sev   = xgb.DMatrix(X_sev_val, label=y_sev_val)
dtest_sev  = xgb.DMatrix(X_sev_test, label=y_sev_test)

def objective_xgb_sev(trial):
    params = {
        'objective': 'reg:gamma',
        'tree_method': 'hist',
        'device': 'cuda',
        'eval_metric': 'gamma-nloglik',
        'max_depth': trial.suggest_int('max_depth', 2, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 50),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
    }

    model = xgb.train(
        params, dtrain_sev,
        num_boost_round=500,
        evals=[(dval_sev, 'val')],
        early_stopping_rounds=30,
        verbose_eval=False
    )
    pred_val = model.predict(dval_sev)
    return np.mean(np.abs(y_sev_val - pred_val))  # MAE, comparable au GLM

study_xgb_sev = optuna.create_study(direction='minimize')
study_xgb_sev.optimize(objective_xgb_sev, n_trials=25, show_progress_bar=True)

print(f"Meilleur MAE (val) : {study_xgb_sev.best_value:.2f}")
print(f"Meilleurs params : {study_xgb_sev.best_params}")

  0%|          | 0/25 [00:00<?, ?it/s]

Meilleur MAE (val) : 1027.62
Meilleurs params : {'max_depth': 6, 'learning_rate': 0.13739022274789708, 'subsample': 0.8938845927961958, 'colsample_bytree': 0.8370607390083245, 'min_child_weight': 12, 'reg_alpha': 0.023299595110562246, 'reg_lambda': 0.15687959894429557}


In [11]:
# ============================================================
# BLOC 7 — RÉENTRAÎNEMENT FINAL XGBOOST SÉVÉRITÉ
# ============================================================
best_params_xgb_sev = study_xgb_sev.best_params.copy()
best_params_xgb_sev.update({
    'objective': 'reg:gamma',
    'tree_method': 'hist',
    'device': 'cuda',
    'eval_metric': 'gamma-nloglik',
})

xgb_sev_final = xgb.train(
    best_params_xgb_sev, dtrain_sev,
    num_boost_round=1000,
    evals=[(dval_sev, 'val')],
    early_stopping_rounds=30,
    verbose_eval=False
)

pred_xgb_sev_test = xgb_sev_final.predict(dtest_sev)
mae_xgb_sev = np.mean(np.abs(y_sev_test - pred_xgb_sev_test))

print(f"Nombre d'arbres retenus : {xgb_sev_final.best_iteration}")
print(f"MAE XGBoost sévérité (test) : {mae_xgb_sev:.2f} €")
print(f"Coût moyen observé  : {y_sev_test.mean():.2f} €")
print(f"Coût moyen prédit   : {pred_xgb_sev_test.mean():.2f} €")

Nombre d'arbres retenus : 7
MAE XGBoost sévérité (test) : 1027.62 €
Coût moyen observé  : 1565.75 €
Coût moyen prédit   : 1483.11 €


In [14]:
# ============================================================
# BLOC 8 — LIGHTGBM GAMMA (SÉVÉRITÉ) AVEC OPTUNA
# ============================================================
dtrain_sev_lgb = lgb.Dataset(X_sev_tr, label=y_sev_tr)
dval_sev_lgb   = lgb.Dataset(X_sev_val, label=y_sev_val, reference=dtrain_sev_lgb)

def objective_lgb_sev(trial):
    params = {
        'objective': 'gamma',
        'metric': 'rmse',
        'verbosity': -1,
        'feature_pre_filter': False,
        'max_depth': trial.suggest_int('max_depth', 2, 8),
        'num_leaves': trial.suggest_int('num_leaves', 7, 127),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': 1,
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-3, 10, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-3, 10, log=True),
    }
    model = lgb.train(
        params, dtrain_sev_lgb,
        num_boost_round=500,
        valid_sets=[dval_sev_lgb],
        callbacks=[lgb.early_stopping(30, verbose=False)]
    )
    pred_val = model.predict(X_sev_val)
    return np.mean(np.abs(y_sev_val - pred_val))

study_lgb_sev = optuna.create_study(direction='minimize')
study_lgb_sev.optimize(objective_lgb_sev, n_trials=25, show_progress_bar=True)

best_params_lgb_sev = study_lgb_sev.best_params.copy()
best_params_lgb_sev.update({'objective': 'gamma', 'metric': 'rmse', 'verbosity': -1})

lgb_sev_final = lgb.train(
    best_params_lgb_sev, dtrain_sev_lgb,
    num_boost_round=1000,
    valid_sets=[dval_sev_lgb],
    callbacks=[lgb.early_stopping(30, verbose=False)]
)

pred_lgb_sev_test = lgb_sev_final.predict(X_sev_test)
mae_lgb_sev = np.mean(np.abs(y_sev_test - pred_lgb_sev_test))
print(f"MAE LightGBM sévérité (test) : {mae_lgb_sev:.2f} €")
print(f"Coût moyen prédit LightGBM   : {pred_lgb_sev_test.mean():.2f} €")

  0%|          | 0/25 [00:00<?, ?it/s]

MAE LightGBM sévérité (test) : 1050.52 €
Coût moyen prédit LightGBM   : 1533.65 €


In [15]:
# ============================================================
# BLOC 9 — COMPARAISON SÉVÉRITÉ + SAUVEGARDE
# ============================================================
comparaison_sev = pd.DataFrame({
    'Modèle': ['GLM Gamma', 'XGBoost', 'LightGBM'],
    'MAE test (€)': [1066.26, mae_xgb_sev, mae_lgb_sev],
    'Coût moyen prédit (€)': [1562.31, pred_xgb_sev_test.mean(), pred_lgb_sev_test.mean()],
})
comparaison_sev['Coût moyen observé (€)'] = y_sev_test.mean()
print(comparaison_sev)

# Sauvegarde des modèles + hyperparamètres optimaux
os.makedirs(f'{BASE}/models', exist_ok=True)

xgb_freq_final.save_model(f'{BASE}/models/xgb_freq.json')
xgb_sev_final.save_model(f'{BASE}/models/xgb_sev.json')

with open(f'{BASE}/models/lgb_freq.pkl', 'wb') as f:
    pickle.dump(lgb_freq_final, f)
with open(f'{BASE}/models/lgb_sev.pkl', 'wb') as f:
    pickle.dump(lgb_sev_final, f)

best_params_all = {
    'xgb_freq': best_params_xgb_freq,
    'lgb_freq': best_params_lgb_freq,
    'xgb_sev': best_params_xgb_sev,
    'lgb_sev': best_params_lgb_sev,
}
with open(f'{BASE}/models/best_params.json', 'w') as f:
    json.dump(best_params_all, f, indent=2)

print("\nModèles XGBoost et LightGBM sauvegardés.")

      Modèle  MAE test (€)  Coût moyen prédit (€)  Coût moyen observé (€)
0  GLM Gamma   1066.260000            1562.310000             1565.750798
1    XGBoost   1027.615795            1483.113037             1565.750798
2   LightGBM   1050.521340            1533.654373             1565.750798

Modèles XGBoost et LightGBM sauvegardés.


In [16]:
# ============================================================
# BLOC 10 — RECALIBRATION (BALANCE CORRECTION) — SÉVÉRITÉ
# ============================================================
# Pratique standard : on applique un facteur de correction
# multiplicatif pour que la moyenne prédite = moyenne observée
# sur le train (PAS sur le test, pour éviter le data leakage)

pred_xgb_sev_train = xgb_sev_final.predict(dtrain_sev)
pred_lgb_sev_train = lgb_sev_final.predict(X_sev_tr)

correction_xgb = y_sev_tr.mean() / pred_xgb_sev_train.mean()
correction_lgb = y_sev_tr.mean() / pred_lgb_sev_train.mean()

print(f"Facteur de correction XGBoost : {correction_xgb:.4f}")
print(f"Facteur de correction LightGBM : {correction_lgb:.4f}")

# Application sur le test
pred_xgb_sev_test_calib = pred_xgb_sev_test * correction_xgb
pred_lgb_sev_test_calib = pred_lgb_sev_test * correction_lgb

mae_xgb_sev_calib = np.mean(np.abs(y_sev_test - pred_xgb_sev_test_calib))
mae_lgb_sev_calib = np.mean(np.abs(y_sev_test - pred_lgb_sev_test_calib))

print(f"\nMAE XGBoost recalibré : {mae_xgb_sev_calib:.2f} € (vs {mae_xgb_sev:.2f} brut)")
print(f"MAE LightGBM recalibré : {mae_lgb_sev_calib:.2f} € (vs {mae_lgb_sev:.2f} brut)")
print(f"\nCoût moyen prédit XGBoost recalibré  : {pred_xgb_sev_test_calib.mean():.2f} €")
print(f"Coût moyen prédit LightGBM recalibré : {pred_lgb_sev_test_calib.mean():.2f} €")
print(f"Coût moyen observé                    : {y_sev_test.mean():.2f} €")

Facteur de correction XGBoost : 1.0508
Facteur de correction LightGBM : 1.0202

MAE XGBoost recalibré : 1070.28 € (vs 1027.62 brut)
MAE LightGBM recalibré : 1069.35 € (vs 1050.52 brut)

Coût moyen prédit XGBoost recalibré  : 1558.40 €
Coût moyen prédit LightGBM recalibré : 1564.67 €
Coût moyen observé                    : 1565.75 €
